In [28]:
import os
import numpy as np
import pandas as pd
import neo
import matplotlib.pyplot as plt
from tqdm import tqdm
import gc

In [29]:
abf_path = r"G:\高醫心肌梗塞_dataset_20250725\OneDrive_1_2025-7-25\MACE_SKNA/"
hp_id_path = r"V:\dunwei\MACE\dataset\SKNA_signal/"
save_path = r"V:\dunwei\MACE\dataset\SKNA_signal/"

In [27]:
if os.path.exists(save_path) is False:
    os.makedirs(save_path)
    print(f"Create {save_path} successfully!")
else:
    print(f"{save_path} already exists!")

V:\dunwei\MACE\dataset\SKNA_signal/ already exists!


In [28]:
h_id = pd.read_csv(os.path.join(hp_id_path, 'MACE_h_id.csv'))
p_id = pd.read_csv(os.path.join(hp_id_path, 'MACE_p_id.csv'))

hp_df = pd.concat([h_id, p_id], axis=0, ignore_index=True)
hp_id = hp_df['research_ID'].tolist()
# print(hp_id)
print(len(hp_id))

524


In [29]:
subjects_time = pd.DataFrame()
error_id = pd.DataFrame()
for i in tqdm(range(len(hp_id)), desc="Processing ABF files"):
    try:
        abf_file = os.path.join(abf_path,str(hp_id[i]) + '.abf')
        if not os.path.exists(abf_file):
            print(f"File {abf_file} does not exist, skipping...")
            error_id = pd.concat([error_id, pd.DataFrame({'research_ID': [hp_id[i]], 'reason': 'File does not exist'})],ignore_index=True)
            continue
        
        reader = neo.io.AxonIO(abf_file)
        block = reader.read_block()
        ch1_signal = block.segments[0].analogsignals[0]
        ch1_signal_second = ch1_signal.duration.rescale('s').magnitude
        ch1_signal_minute = ch1_signal_second / 60
        subjects_time = pd.concat([subjects_time, pd.DataFrame({'research_ID': [hp_id[i]], 'minutes': [np.round(ch1_signal_minute, 2)], 'seconds': [np.round(ch1_signal_second, 2)]})], ignore_index=True)
        
    except Exception as e:
        print(f"Error processing file {abf_file}: {e}")
        error_id = pd.concat([error_id, pd.DataFrame({'research_ID': [hp_id[i]], 'reason': 'Error processing file'})],ignore_index=True)
        continue

    del reader, block
    gc.collect()

print(subjects_time)
print(error_id)

subjects_time.to_csv(os.path.join(save_path, 'subjects_time.csv'), index=False)
error_id.to_csv(os.path.join(save_path, 'error_subjects_time.csv'), index=False)

    

Processing ABF files:  62%|██████▏   | 327/524 [00:45<00:34,  5.79it/s]

Error processing file G:\高醫心肌梗塞_dataset_20250725\OneDrive_1_2025-7-25\MACE_SKNA/534458973157.abf: 'NoneType' object is not subscriptable


Processing ABF files:  81%|████████  | 424/524 [01:01<00:11,  8.95it/s]

Error processing file G:\高醫心肌梗塞_dataset_20250725\OneDrive_1_2025-7-25\MACE_SKNA/574558462689.abf: 'NoneType' object is not subscriptable


Processing ABF files: 100%|██████████| 524/524 [01:15<00:00,  6.95it/s]

      research_ID  minutes  seconds
0    414490914181    21.79  1307.64
1    534491345995    16.77  1006.39
2    584487583289    15.15   909.13
3    354480714133    17.38  1042.64
4    364481044155    18.07  1084.38
..            ...      ...      ...
517  514556714991    17.92  1075.30
518  554556896417    25.46  1527.55
519  464557487015    22.04  1322.54
520  534557456881    18.06  1083.68
521  624558969781    15.32   919.33

[522 rows x 3 columns]
    research_ID                 reason
0  534458973157  Error processing file
1  574558462689  Error processing file
